# The predictive self-check co-processor

A controller's body model knows the robot's *mechanics* — masses, joints,
kinematics. But a specific unit also has **off-body physics** the body model never
sees: its servos heat up, its battery sags under load. A command that looks fine to
the body model can cook a motor or brown out the pack a second later.

The fix is a **per-unit twin** running as a co-processor *beside* the controller.
Each cycle it forecasts the unit's near future — predicted servo temperature,
predicted bus voltage — and a `SafetyMonitor` **vetoes or derates** commands that
would breach a limit, *before* they're sent. This is the catch: the body-only
model says *proceed*; the twin says *that will overheat in N seconds*.

```
   controller ─► command ─┐
                          ├─► twin co-processor ─► forecast ─► SafetyMonitor ─► OK / DERATE / VETO
   state ─────────────────┘   (body + servo-thermal + battery)   (per-unit limits)
```

This tutorial builds a LeKiwi twin, forecasts under load, and shows two checks: a
**thermal** veto and the LeKiwi-specific **brownout** veto.

In [1]:
import logging, warnings
for _n in ("jaxonomy","jaxterity","mujoco","absl"): logging.getLogger(_n).setLevel(logging.ERROR)
logging.disable(logging.WARNING); warnings.filterwarnings("ignore")
import numpy as np, jax.numpy as jnp
from jaxterity.twin import lekiwi_twin
from jaxterity.selfcheck import (CoProcessor, SafetyMonitor, SafetyEnvelope, SafetyAction,
    preflight, constant_policy, Scenario)
print("imports OK")

imports OK


## Step 1 — The per-unit twin

`lekiwi_twin(serial)` composes three models for *this* unit: the calibrated **body**
(MJX articulated dynamics), a per-servo **thermal** model (winding temperature
heated by a torque²-proxy), and — because the LeKiwi shares one battery across all
servos — a **battery-sag** model (bus voltage drops under aggregate current). The
serial makes it a specific unit, not a generic design.

In [2]:
twin = lekiwi_twin("LK-0001")
nq, nv = twin.robot.build_system()._nq, twin.n_servos
print("twin serial    :", twin.serial)
print("servos         :", nv)
print("has battery    :", twin.has_battery, "| V_oc =", twin.V_oc, "V | R_int =", twin.R_int, "Ω")
print("ambient (T_amb):", twin.T_amb, "K (", round(twin.T_amb-273.15,1), "°C )")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


twin serial    : LK-0001
servos         : 6
has battery    : True | V_oc = 12.0 V | R_int = 0.3 Ω
ambient (T_amb): 298.15 K ( 25.0 °C )


## Step 2 — Forecast the near future

The `CoProcessor` rolls the twin forward a short horizon and publishes the
predicted trajectories. Under a **heavy** sustained load you can already see where
it's heading: the bus voltage sags and the winding temperature climbs.

In [3]:
cp = CoProcessor(twin, rate_hz=20.0, horizon_steps=8, dt=0.02)
q0, qd0, T0 = jnp.zeros(nq), jnp.zeros(nv), jnp.full(nv, twin.T_amb)
heavy = jnp.full(nv, 2.0)   # N·m on every servo — a sustained heavy hold
pred = cp.predict(q0, qd0, T0, heavy)
print("predicted bus voltage over the horizon (V):",
      [round(float(v),2) for v in pred.bus_voltage])
print("predicted servo-0 temperature (°C):",
      [round(float(t)-273.15,1) for t in pred.temperature[:, 0]])

predicted bus voltage over the horizon (V): [6.6, 6.6, 6.6, 6.6, 6.6, 6.6, 6.6, 6.6]
predicted servo-0 temperature (°C): [25.0, 25.0, 25.1, 25.1, 25.1, 25.2, 25.2, 25.2, 25.3]


## Step 3 — The thermal check: body proceeds, the twin vetoes

Here is the whole point. The body-only model sees torque available and says
*proceed*. The twin forecasts that the same sustained hold drives the servo
winding past its temperature ceiling — so the monitor **vetoes**, names the binding
constraint, and even estimates the time to breach.

In [4]:
# thermal ceiling 30 K above ambient; bus floor out of the way so THERMAL is the test
thermal_env = SafetyEnvelope(temp_ceiling_k=twin.T_amb + 30.0, bus_floor_v=0.0)
mon = SafetyMonitor(twin, thermal_env)

heavy_decision = mon.evaluate(cp.predict(q0, qd0, T0, heavy), action=heavy)
light = jnp.full(nv, 0.3)
light_decision = mon.evaluate(cp.predict(q0, qd0, T0, light), action=light)

print("body-only model would say: torque is available -> PROCEED (both commands)\n")
print("twin co-processor:")
print(f"  heavy hold -> {heavy_decision.action.name:5s}  binding={heavy_decision.binding_constraint}"
      f"  time_to_breach={heavy_decision.time_to_breach_s}")
print(f"  light hold -> {light_decision.action.name:5s}  (within the unit's thermal envelope)")

body-only model would say: torque is available -> PROCEED (both commands)

twin co-processor:
  heavy hold -> VETO   binding=servo_temperature  time_to_breach=22.48160696648464
  light hold -> OK     (within the unit's thermal envelope)


## Step 4 — The LeKiwi brownout: the shared-battery axis

A bench arm only worries about heat. The LeKiwi shares **one battery** across nine
servos, so aggregate load is its own failure mode. With a bus-voltage floor in the
envelope, the twin vetoes a heavy command whose predicted bus voltage sags below
it — a brownout caught before the pack actually droops.

In [5]:
brownout_env = SafetyEnvelope(temp_ceiling_k=twin.T_amb + 500.0,  # thermal out of the way
                              bus_floor_v=10.5, bus_band_v=0.5)
bmon = SafetyMonitor(twin, brownout_env)

bh = bmon.evaluate(cp.predict(q0, qd0, T0, heavy), action=heavy)
bl = bmon.evaluate(cp.predict(q0, qd0, T0, light), action=light)
print(f"heavy load -> {bh.action.name:5s}  binding={bh.binding_constraint}"
      f"  worst bus voltage={bh.worst_value:.2f} V  (floor 10.5 V)")
print(f"light load -> {bl.action.name:5s}  (bus stays above the floor)")

heavy load -> VETO   binding=bus_voltage  worst bus voltage=6.60 V  (floor 10.5 V)
light load -> OK     (bus stays above the floor)


## Step 5 — Pre-flight: check a policy *before* it runs

The same envelope powers an offline gate. `preflight` simulates a candidate policy
on the twin across declared scenarios and passes/fails it *before deployment* —
so an unsafe controller never reaches the robot. It also records the twin's
attestation handle, so a pass binds to a specific unit + model version.

In [6]:
scenario = Scenario(name="sustained-hold", q0=np.zeros(nq), qd0=np.zeros(nv),
                    T0=np.full(nv, float(twin.T_amb)), n_steps=20, dt=0.02)
env = SafetyEnvelope(temp_ceiling_k=twin.T_amb + 30.0, bus_floor_v=10.5, bus_band_v=0.5)

heavy_report = preflight(constant_policy(np.full(nv, 2.0)), twin, env, [scenario])
light_report = preflight(constant_policy(np.full(nv, 0.3)), twin, env, [scenario])
print("preflight a heavy-hold policy :", "PASS" if heavy_report.passed else "FAIL (rejected before deploy)")
print("preflight a light-hold policy :", "PASS" if light_report.passed else "FAIL")

preflight a heavy-hold policy : FAIL (rejected before deploy)
preflight a light-hold policy : PASS


## Recap

The self-check co-processor is the **predictive safety layer** for a specific unit:

- a **per-unit twin** = the calibrated body + off-body physics (servo thermal +
  LeKiwi battery sag) the body model never sees;
- a **co-processor** that forecasts the near future beside the controller;
- a **`SafetyMonitor`** that vetoes/derates a command the body model would pass —
  on the **thermal** axis *and* the LeKiwi **brownout** axis — naming the binding
  constraint and the time to breach;
- and **pre-flight**, the same envelope as an offline gate that rejects an unsafe
  policy before it ever reaches the robot.

It complements the dual-path arbiter (tutorial #3): the arbiter guards the *control
loop* with a fallback; the self-check guards the *unit's physics* with a forecast.
Together they're how a learned, deployed controller stays safe on real hardware.